### CNN For Cats and Dogs Dataset 

In [46]:
import torch 
import torch.nn as nn
import torchvision

In [47]:
import torch.optim as optim


### Dataset and DataLoader 

In [48]:
from torch.utils.data import DataLoader 
import torchvision.transforms as transforms

In [49]:
transform=transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize((0.5),(0.5))
    ]
)

In [50]:
from PIL import Image
import os

dataset_path = "data/PetImages"

corrupted = 0

for folder in ["Cat", "Dog"]:
    folder_path = os.path.join(dataset_path, folder)

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        try:
            img = Image.open(file_path)
            img.verify()      # Check if image is valid

        except Exception:
            print("Removing:", file_path)
            os.remove(file_path)
            corrupted += 1

print(f"Total corrupted images removed: {corrupted}")

Removing: data/PetImages\Cat\666.jpg
Removing: data/PetImages\Cat\Thumbs.db
Removing: data/PetImages\Dog\11702.jpg
Removing: data/PetImages\Dog\Thumbs.db
Total corrupted images removed: 4


In [51]:
from torchvision import datasets 

dataset=datasets.ImageFolder(
    root="data/PetImages",
    transform=transform
)

In [52]:
dataset

Dataset ImageFolder
    Number of datapoints: 24998
    Root location: data/PetImages
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [53]:
from torch.utils.data import random_split

train_size=int(0.8*len(dataset))
test_size=int(len(dataset)-train_size)

trainset,testset=random_split(
    dataset,
    [train_size,test_size]
)

In [54]:
print(len(trainset))

19998


In [55]:
print(len(testset))

5000


In [56]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

### Defining CNN 

In [57]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(16*16*128,256),
            nn.ReLU(),
            nn.Linear(256,2)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x 

In [58]:
model=CNN()

In [59]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### Training the CNN

In [60]:
epochs=10

for epoch in range(epochs):

    model.train()

    training_loss=0.0

    for images,labels in trainloader:

        optimizer.zero_grad()

        outputs=model(images)
        loss=criteria(outputs,labels)
        loss.backward()
        optimizer.step()

        training_loss+=loss.item()

    print(f"{epoch+1} , loss={training_loss/len(trainloader)}")


1 , loss=0.6035082894392287
2 , loss=0.47831550516640414
3 , loss=0.3942192447737764
4 , loss=0.32258193073466945
5 , loss=0.23885552587505346
6 , loss=0.14605788331919203
7 , loss=0.08459876568172686
8 , loss=0.04108174953359765
9 , loss=0.023862793909968993
10 , loss=0.022115466154229563
